# Initialization

## Path configuration

In [ ]:
# Path configuration
import sys
from pathlib import Path

# Navigate up from 'src/notebooks/' to the project root directory
PROJECT_ROOT = Path("..").resolve().parent
DATA_DIR = PROJECT_ROOT / "data"
DOCS_DIR = PROJECT_ROOT / "docs"
SRC_DIR = PROJECT_ROOT / "src"

# Add src to python path for imports
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

# Enable automatic reloading (avoid restarting kernel when editing internal modules)
%reload_ext autoreload
%autoreload 2

## Imports

In [ ]:
from typing import List

import pandas as pd
import numpy as np

import plotly.io as pio

import statsmodels.api as sm
import statsmodels.formula.api as smf

import xgboost as xgb

from sklearn.metrics import mean_poisson_deviance
from sklearn.model_selection import RandomizedSearchCV, KFold

from utils.plots import plot_lorenz_curves, plot_double_lift_chart

In [ ]:
# Set renderer for plots
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Predictive Modeling

We begin by reading the training, validation and test data sets that we created in Part I:

In [ ]:
df_train = pd.read_csv(DOCS_DIR / "train.csv")
df_val = pd.read_csv(DOCS_DIR / "val.csv")
df_test = pd.read_csv(DOCS_DIR / "test.csv")

We also stablish what features will be used for training as per the Exploratory Data Analysis (EDA) that we performed in Part I:

In [ ]:
# Final list of features to be included
categorical_features = ["VehPower", "VehBrand", "VehGas", "Region"]  # Area is dropped as per the EDA
numerical_features = [
    "VehAge",
    "DrivAge",
    "BonusMalus",
    "log_density",
]  # Density replaced by log_density as per the EDA
actuarial_features = ["is_malus", "is_young_driver", "is_new_car", "is_short_term"]
features = categorical_features + numerical_features + actuarial_features

## Model selection

In order to predict the number of claims (`ClaimNb`), we will test two distinct approaches:

1. **Poisson GLM** (with log-link function)
2. **XGBoost** (with Poisson objective)

While previously hinted at, **Poisson** models are the most suitable for this task: our goal is to count rare events, which is exactly what the Poisson distribution was designed for. As seen in the preprocessing phase, 95% of policies have zero claims, with counts declining rapidly; a distribution that fits a Poisson shape perfectly.

The EDA demonstrated that claim counts are tightly linked with `Exposure`, which defines the frequency of claims. This justifies using **exposure as an offset variable** in both models.

#### GLM

* **Justification:** Claim counts are discrete and sparse, making the Poisson distribution the natural statistical choice for frequency modeling.
* **Assumption:** The log-link function is used because insurance risk factors are typically multiplicative. The GLM allows us to quantify the exact "relativity" (e.g., a +15% risk increase) for each feature.
* **Strengths:** High transparency, regulatory acceptance, and mathematical stability.
* **Weaknesses:** It assumes features act independently and cannot detect complex interactions without manual intervention (e.g., the combined effect of high age and high malus).

#### XGBoost

* **Justification:** As a tree-based ensemble, XGBoost is "model-agnostic" regarding the data's shape and does not assume a linear relationship between log-frequency and features.
* **Capturing Interactions:** Its primary value lies in automatically discovering non-linear interactions and "local" patterns that a GLM would overlook.
* **Consistency:** By using the `count:poisson` objective and the same offset, we ensure a "fair fight" against the GLM, isolating the performance gain attributable to the algorithm's flexibility.

#### Why not Tweedie?

While Tweedie regression was considered, our target is the number of claims (frequency) rather than total cost (severity). Since counts are integers, the Poisson distribution (a special case of the Tweedie family where power p=1) is the most theoretically appropriate and parsimonious choice.

#### Benchmarking

It is good practice to use a simple model for benchmarking to quantify the value added by sophisticated algorithms. This also controls for preparation errors: if results are worse than a "dummy" model, a technical issue is likely. Our benchmark assigns the **average claim frequency** to every policy, essentially assuming "all drivers are equal."

#### Evaluation Metric

To evaluate performance, we must measure how much predictions deviate from actual values. We will use **Mean Poisson Deviance**, which is the standard loss metric for models fitting Poisson distributions.

## Benchmark model

In [ ]:
# Make predictions with benchmark model (assign average frequency to all drivers)
train_freq = df_train["ClaimNb"].sum() / df_train["exposure_stable"].sum()
df_train["bmk_pred"] = np.full(len(df_train), train_freq * df_train["exposure_stable"])
df_test["bmk_pred"] = np.full(len(df_test), train_freq * df_test["exposure_stable"])

# Benchmark reference
bmk_ref = mean_poisson_deviance(df_train["ClaimNb"], df_train["bmk_pred"])

# Benchmark performance
bmk_dev = mean_poisson_deviance(df_test["ClaimNb"], df_test["bmk_pred"])

print(f"Benchmark reference for training: {bmk_ref:.4f}")
print(f"Benchmark performance (test set): {bmk_dev:.4f}")

## GLM

### Training

In [ ]:
# Fit the Poisson GLM
glm_results = smf.glm(
    formula="ClaimNb ~ " + " + ".join(features),
    data=df_train,
    family=sm.families.Poisson(),
    offset=np.log(df_train["exposure_stable"]),
).fit(cov_type="HC3")

### Evaluation

In [ ]:
print(glm_results.summary())

In [ ]:
df_train["glm_pred"] = glm_results.predict(df_train, offset=np.log(df_train["exposure_stable"]))

glm_ref = mean_poisson_deviance(df_train["ClaimNb"], df_train["glm_pred"])

print(f"Poisson deviance on training set: {glm_ref:.4f}")
print(f"Improvement over mean frequency: {(1 - glm_ref / bmk_ref) * 100:.2f}%")

**Model Fit**

The Poisson GLM reached convergence after 7 iterations. On the training set, the model achieved a mean Poisson deviance of 0.3076, capturing 5.40% of the deviance relative to the benchmark. The scale parameter was maintained at 1.0, ensuring the model adheres to the strict Poisson assumption required for a standard frequency rating structure.

**Statistical significance of engineered features**

- All engineered binary indicators —`is_malus`, `is_young_driver`, `is_new_car`, and `is_short_term`— yielded p-values of 0.000. This confirms that these features capture distinct risk discontinuities that a standard continuous model might smooth over.

**Main risk drivers**

Since the model uses a log-link, we can interpret the coefficients as multiplicative effects by exponentiating them:

   - `is_new_car` (*exp(0.9586)=2.6080*): Brand-new cars show a very high relative risk indicator at **+161%** compared to older cars. This suggests that new vehicles in this dataset are significantly more likely to report claims. This aligns with what we saw in the EDA.

   - `is_short_term` (*exp(0.8935)=2.4436*): Short-term policies are associated with a nearly **+144%** increase in frequency compared to standard annual policies, confirming they represent a much higher-churn, higher-risk segment.

   - `is_malus` (*exp(0.2153)=1.2402*): The split bonus/malus indicates that drivers with malus have a **+24%** claim frequency compared to the ones with bonus (note that each additional point in the `BonusMalus` score increases the expected claim frequency by approximately +2.1% as per its own coefficient)

   - `is_young_driver` (*exp(0.1174)=1.1246*): Even after controlling for the continuous `DrivAge`, being in the 18–25 bracket adds an additional **+12%** risk.

**Geographic & Technical Factors**

  - `log_density` (*exp(0.0406)=1.0414*): As expected, higher population density correlates with higher frequency. The statistical relevance indicated by the p-value of 0.000 justifies our decision to keep this as a continuous predictor over the redundant `Area` categories.

  - `Region[T.R94]`(*exp(0.1987)=1.2198*): It is important to understand this example because region R94 exhibits one of the highest relativities in the model, with an estimated +22% increase in claim frequency relative to the baseline region. However, as noted in the EDA, R94 represents a small fraction of the total exposure. While the model captures this high observed frequency, the result should be interpreted with caution due to the limited sample size, which may lead to higher volatility in the estimate compared to high-exposure regions. Similarly, compared to other metrics that affect all drivers (e.g. everyone has a BonusMalus scoring but almost no one is in region R94), the importance of a single small region is limited.

  - `VehGas[T.Regular]` (*exp(0.0569)=1.0585*): Regular fuel types are statiscally significant (p-value=0.000) and show a **+5.9%** higher risk compared to the diesel. This an effect that was no apparent to us from the EDA alone. Qualitatively speaking, this figure may indicate that each type of car is generally utilised in different conditions or with different purposes, which may expose them differently to potential risks.

### Performance

In [ ]:
# Predict on test set
df_test["glm_pred"] = glm_results.predict(df_test, offset=np.log(df_test["exposure_stable"]))

#### Calibration

In [ ]:
# Check there is no drift
print(f"Actual claims in test: {df_test['ClaimNb'].sum()}")
print(f"GLM predicted claims: {df_test['glm_pred'].sum():.2f}")

The fact that the total number of claims is properly matched shows that the model is mathematically sound and that using `Exposure` as an offset was the right move. Moreover the model is unbiased, meaning that if we were to use it for pricing, we would collect exactly enough premium to cover the predicted claims.

#### Deviance

In [ ]:
# Compute mean Poisson deviance
glm_dev = mean_poisson_deviance(df_test["ClaimNb"], df_test["glm_pred"])

print(f"Mean Poisson deviance: {glm_dev:.4f}")
print(f"Improvement over benchmark: {((1 - glm_dev / bmk_dev) * 100):.2f}%")

We improved deviance by 5.26% with respect to the benchmark model. This confirms that the GLM model is strong and it means our features are capturing real, actionable risk signals.

## XGBoost

### Data preparation

In [ ]:
# Convert to One-Hot Encoding
X_train = pd.get_dummies(df_train[features], drop_first=True)
X_val = pd.get_dummies(df_val[features], drop_first=True)
X_test = pd.get_dummies(df_test[features], drop_first=True)

# Align columns
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Create DMatrix structures for efficiency
dtrain = xgb.DMatrix(X_train, label=df_train["ClaimNb"])
dval = xgb.DMatrix(X_val, label=df_val["ClaimNb"])
dtest = xgb.DMatrix(X_test, label=df_test["ClaimNb"])

# base_margin is the offset
dtrain.set_base_margin(np.log(df_train["exposure_stable"]))
dval.set_base_margin(np.log(df_val["exposure_stable"]))
dtest.set_base_margin(np.log(df_test["exposure_stable"]))

### Hyperparameter tuning

Before running the model we perform hyperparameter selection by using randomized search with k-fold cross-validation. This is faster than grid search and simpler than Bayeasian search. It can also focus on more varied values of the most important parameters. The cross-validation ensures that the chosen parameters work consistenly across different subsets of the data.

In [ ]:
# Define the model using the scikit-learn wrapper
# (necessary to use sklearn's SearchCV tools)
xgb_model = xgb.XGBRegressor(
    objective="count:poisson",
    tree_method="hist",
    n_estimators=150,  # A moderate number for the search phase
    n_jobs=-1,
)

# Search space
param_grid = {
    "max_depth": [3, 4, 5, 6, 7],  # How complex are the interactions?
    "learning_rate": [0.01, 0.05, 0.1],  # How fast does it learn?
    "subsample": [0.7, 0.8, 0.9, 1.0],  # % of rows used per tree
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],  # % of features used per tree
    "reg_lambda": [1, 5, 10],  # L2 regularization
    "reg_alpha": [0, 1, 5],  # L1 regularization
    "gamma": [0, 0.5, 1, 2],  # Prevents creating branches that are not informative
}

# Setup K-Fold and RandomizedSearch
kf = KFold(n_splits=5, shuffle=True, random_state=13)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=15,  # How many random combinations to try
    scoring="neg_mean_poisson_deviance",  # We want to minimize Poisson deviance
    cv=kf,
    verbose=1,
    random_state=13,
    n_jobs=-1,  # Use all CPU cores
)

# Execute the search
random_search.fit(
    X_train,
    df_train["ClaimNb"],
    base_margin=np.log(df_train["exposure_stable"]),  # The sklearn wrapper of XGBoost requires passing the base_margin
)

print(f"Best Parameters: {random_search.best_params_}")

The combination of parameters that we got is safe and conservative set of parameters, which is what we need in insurance modeling to ensure the model generalizes well to new customers.

Let us briefly evaluate the results:

- `learning_rate: 0.01`, `max_depth: 6`: A very low learning rate combined with a moderate depth indicates a slow and steady learning process. This prevents the model from being overly influenced by individual unlucky observations, building a more robust consensus across trees.

- `reg_lambda: 5`, `reg_alpha: 5`, `gamma: 1`: These are relatively high values for L1 and L2 regularization. They penalize overly complex trees, forcing the model to only keep features that provide a significant, consistent boost to predictive power.

- `subsample: 0.7`: By training each tree on only 70% of the data, the model introduces "noise" that actually makes the final ensemble more resilient to outliers and sampling bias.

- `colsample_bytree: 1.0`: The model uses all available features for every tree. This suggests that in our specific dataset, the engineered features (like `is_young_driver` or `is_malus`) are distinct enough that the model doesn't need to hide them to find variety.

These results suggest that the XGBoost model is not just memorizing the training data. Because the learning rate is so low and the regularization is so high, the contest between XGBoost and GLM will be fair.


### Training

In [ ]:
params = {
    "objective": "count:poisson",
    "tree_method": "hist",  # Fast training for large data
    "eval_metric": "poisson-nloglik",  # The Poisson equivalent of Log-Loss
    "seed": 13,  # We want reproducible results
} | random_search.best_params_

In [ ]:
# Train with early stopping to prevent overfitting
xgb_results = xgb.train(
    params,
    dtrain,
    num_boost_round=5000,  # Large number or rounds because learning rate is small
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=50,
    verbose_eval=250,
)

In [ ]:
df_train["xgb_pred"] = xgb_results.predict(dtrain)

xgb_ref = mean_poisson_deviance(df_train["ClaimNb"], df_train["xgb_pred"])

print(f"Poisson deviance on training set: {xgb_ref:.4f}")
print(f"Improvement over mean frequency: {(1 - xgb_ref / bmk_ref) * 100:.2f}%")

The XGBoost model utilized a conservative learning rate (0.01) and early stopping with a 50-round patience window to ensure the ensemble prioritized generalizable patterns over training-set noise. It reached convergence after 4,114 iterations. The model achieved a Mean Poisson Deviance of 0.2860 on the training set, which captures 12.04% of variance with respect to the benchmark. The high number of boosting rounds required for convergence indicates that the model successfully navigated a complex loss surface to capture subtle, non-linear interactions across the feature set that are not accessible via standard additive modeling.

### Feature importance

In [ ]:
def create_feature_importance_df(xgb_results: xgb.core.Booster) -> pd.DataFrame:
    # Gain scores
    dict_scores = xgb_results.get_score(importance_type="gain")

    # Convert to DataFrame
    df_importance = pd.DataFrame(list(dict_scores.items()), columns=["feature", "gain"]).sort_values(
        by="gain", ascending=False
    )

    # Calculate relative importance (%)
    total_gain = df_importance["gain"].sum()
    df_importance["importance_pct"] = (df_importance["gain"] / total_gain) * 100

    # Set feature as index for readability
    df_importance.set_index("feature", inplace=True)

    return df_importance


create_feature_importance_df(xgb_results)

The feature importances we are seeing for the XGBoost are very interesting, specially if we compare them to the ones we saw for GLM. Notice the following;

- `is_new_car`, `is_malus`, `is_young_driver` are not present altogether. How is this possible if, especially the first two, were the main features for the GLM model?
  
  - The explanation is that XGBoost is able to split the base features that we used to generate these derived features in a much more specific way.
  
  - Through `VehAge`, `BonusMalus` and `DrivAge` it already finds the binary actuarial values without us having to signal them. That's why it drops the derived features and and keeps only the base.
  
  - Note how both `VehAge` and `BonusMalus` are now high importance features, replicating what `is_new_car` and `is_malus` did for the GLM.

  - Note that this contrasts with the value `colsample_by_tree=1` that we got in parameter selection. However, we also got `alpha=5` for L1 regularization, which is prone to killing unneeded features.

- `is_short_term` is still a high importance feature. Why did it not drop it and use the base instead?
   
   - Because the base feature is `Exposure`, which is *not* a feature of our model, it is just the offset.
   
   - Using `Exposure` as offset has mathematical relevance, but the high importance of `is_short_term` shows that there is more to it: as observed in prior sections, there is an actuarial value that makes drivers with short exposures qualitatively different from other drivers.

- Interestingly `VehBrand_B12` now shows up as a very relevant player. In the EDA we already noticed the fact that it has the highest claim ratio among all brands and is among the most common brands. GLM apparently did not capture this importance too well because other brands had higher coefficients.

- As a final comment we see that `VehGas_Regular` also has high importance. In the GLM evaluation we discussed that it had higher risk than Diesel gas, but it was not placed that high in the feature importance list.

- It is also worth noticing that now `Region_R94` is placed much lower than it was for GLM. That would confirm the concerns we exposed in that section.

### Performance

In [ ]:
# Predict on test set
df_test["xgb_pred"] = xgb_results.predict(dtest)

#### Calibration

In [ ]:
# Check there is no drift
print(f"Actual claims in test set: {df_test['ClaimNb'].sum()}")
print(f"XGBoost predicted claims: {df_test['xgb_pred'].sum():.2f}")

Again the model has almost no drift with respect to the total number of claims, confirming that a Poisson objective with `Exposure` as the offset is indeed the right approach.

#### Deviance

In [ ]:
# Calculate Mean Poisson Deviance
xgb_dev = mean_poisson_deviance(df_test["ClaimNb"], df_test["xgb_pred"])

print(f"Mean Poisson deviance: {xgb_dev:.4f}")
print(f"Improvement over benchmark: {((1 - xgb_dev / bmk_dev) * 100):.2f}%")

XGBoost improves the benchmark by a notable 9.17%. We shall compare it against the GLM in more detail later, but a jump from 5.48% to 9.17% improvement indicates that the model made a good job at capturing relevant non-linear interactions and combinations (e.g. the old_car+malus and old_driver+malus that we observed in the EDA).

However, while the improvement was 12.04% in the training set is now only 9.17% for the test set. For GLM the drop was only from 5.40% to 5.26%. This indicates that XGBoost may be overfitting a bit.

## Comparative evaluation of models

Let us make a reminder on how each of the models performed on the test set with respect to the benchmark model:

In [ ]:
print("Test set scores (mean Poisson deviance)")
print(40 * "=")
print(f"Benchmark   {bmk_dev:.4f}")
print(f"GLM         {glm_dev:.4f} (+{((1 - glm_dev / bmk_dev) * 100):.2f}% improvement)")
print(f"XGBoost     {xgb_dev:.4f} (+{((1 - xgb_dev / bmk_dev) * 100):.2f}% improvement)")

These results indicate that **XGBoost is a winner** at predicting claims for our car insurance claims.

To get a more exhaustive picture on how do the two models compare from an actuarial point of view, we shall use a couple of helpful visualizations.

In [ ]:
plot_lorenz_curves(df_test)

**Lorenz Curve** 

In our context the Lorenz curves aim at showing if a model is good at separating "safe" drivers from "dangerous" ones:

  - **The Diagonal Line:** Represents "random luck". If you pick 20% of drivers at random, you get 20% of the claims.

  - **The Model Curve:** If you rank drivers from highest to lowest risk, a good model will show that the top 20% of people (the ones the model flagged as high risk) actually account for, say, 50% of the claims.

  - **The Result:** As a consequence a curve that bows further away from the diagonal represents a better model.
  
Notice that our benchmark model is already better than a purely radom shuffle. That makes sense because the benchmark assumes "everyone is average", which is better than "anyone is anything". As observed by the scoring, GLM improves over the benchmark and XGBoost improves even more.

To put it in simple terms, the ranking criteria of each model is based on:

  - **Diagonal:** Random Shuffle

  - **Benchmark:** Ranked by Exposure

  - **GLM:** Ranked by Exposure + Linear Features

  - **XGBoost:** Ranked by Exposure + Complex Interactions

In [ ]:
plot_double_lift_chart(df_test)

**Double Lift Chart**

This chart aims at answering a very simple actuarial question: "If my competitor uses a GLM and I use XGBoost, who wins?"

How to read it:

- Going from left to right we move progressively from Bucket 0-where GLM thinks the risk is much higer than XGBoost-to Bucket 9-where XGBoost thinks the risk is much higer than GLM.

- The height of each bar is the actual claim frequency on each bucket.

- The blue and yellow lines are the predicted frequencies for each of the models. A marker above the bar is an overstimate. A marker below the top of the bar is an understimate.

- Therefore, a model performs better if it "hugs" the gray bars (the markers are at a similar height as the top of the bars).

In this regard, XGBoost clearly made a much better job than GLM at capturing the actual frequencies.

## Residual Analysis and Irreducible Risk

Before wrapping up our work we will take a look at residual analysis by inspecting the top 25 errors of the three models (we include the benchmark because it provides useful insights). 

The number 25 is not chosen lightly: this is the exact number of rows in our test set that have 3 claims, corresponding to the capping we put on `ClaimNb`.

In [ ]:
def get_post_mortem_df(df: pd.DataFrame, model_name: str, relevant_features: List[str]) -> pd.DataFrame:
    # Calculate absolute error on the Test Set
    df["error"] = np.abs(df["ClaimNb"] - df[f"{model_name}_pred"])

    # Get the top 10 most "surprising" rows
    top_errors = df.sort_values(by="error", ascending=False).head(25)

    # Set IDpol as index
    top_errors.set_index("IDpol", inplace=True)

    # Display relevant columns to look for patterns
    cols_to_check = ["ClaimNb", f"{model_name}_pred", "Exposure"] + relevant_features

    return top_errors[cols_to_check]

In [ ]:
bmk_pm = get_post_mortem_df(df_test, "bmk", features)
glm_pm = get_post_mortem_df(df_test, "glm", features)
xgb_pm = get_post_mortem_df(df_test, "xgb", features)

In [ ]:
# Count how many errors are shared for each pair of models
print(f"Shared errors in top 25 (Benchmark vs GLM):     {len(set(bmk_pm.index).intersection(set(glm_pm.index)))}")
print(f"Shared errors in top 25 (Benchmark vs XGBoost): {len(set(bmk_pm.index).intersection(set(xgb_pm.index)))}")
print(f"Shared errors in top 25 (GLM vs XGBoost):       {len(set(glm_pm.index).intersection(set(xgb_pm.index)))}")

In [ ]:
# Count how many claims do the top 25 errors have for each model
display(bmk_pm["ClaimNb"].value_counts().to_frame())
display(glm_pm["ClaimNb"].value_counts().to_frame())
display(xgb_pm["ClaimNb"].value_counts().to_frame())

Note that all models report the same top 25 errors and that they all struggle at capturing rows with a top number of claims. This fact should be read from two different angles:

**Model limitations**

A Poisson model (including GLM and XGBoost) predicts the expected value. Even for the riskiest driver in our dataset, the model might only predict relatively low frequency (e.g. the Benchmark predicts a frequency of ~0.1 for all drivers):

  - When that driver has 0 claims in a year, the error is small (0.1).

  - When that driver has 3 claims in a year, the gap is massive (2.9) and even more if exposure was shorter.

Since the models are grounded in reality, they refuse to predict a "3" for anyone, because no one has a *300% probability* of a claim. Therefore, the rows with 3 claims are always the top errors by definition.

**Stochastic noise in the data**

The fact that these are the top 25 errors for *all three models* proves that these 3-claim events are stochastic rather than systematic.

  - If the XGBoost had "fixed" these errors while the GLM didn't, it would mean there was a hidden pattern (e.g., "Drivers from Region X always have 3 claims")
  
  - Since the XGBoost also failed to predict them, it confirms there is no feature in our data (Age, Brand, Gas, etc.) that reliably points to a triple-claim outcome. Moreover, it proves that XGBoost hasn't hallucinated a patter where non exists.

On an extra note, inspecting the errors in more detail reveals that the policies with IDs *2247986*, *2248174* and *2277762* are perfect duplicates (except for their IDs) which might be due to some accidental data entry. In a real world scenario it would be worth double checking extra information on specific policies -if available- to confirm what is the exact situation.

# Conclusions & Recommendations

### Key Findings

* **Performance:** The **XGBoost model** is the clear winner, explaining **9.17%** of the deviance on unseen data, clearly improving the predictive power of the GLM (**5.26%**).

* **Segmentation:** The Lorenz Curve confirms that XGBoost is significantly more capable of isolating high-risk segments, which allows for more competitive pricing and better risk selection.

* **Model Reliability:** While slight overfitting was observed in the XGBoost model, it remains robust. The "top errors" analysis reveals that remaining inaccuracies are due to random claim volatility (3-claim outliers) rather than model failure, affecting all models equally.

### Recommendations

* **Adopt XGBoost for Pricing:** Transition from the GLM to the XGBoost framework to capture non-linear risks that traditional models miss.

* **Monitor "Black Swan" Outliers:** Since 3-claim events are currently unpredictable, consider a separate "Large Loss" loading or reinsurance strategy, as these events are stochastic and not driven by the current feature set.

* **Data Enrichment:** To further close the gap on the 9% test deviance, investigate new features (e.g., annual mileage or telematics) that might explain the high-frequency outliers that the current model classifies as "safe".